# Lab 7: Building Speech-to-Text Applications using GCP API
**Class:** I&C SCI X427.08<br>
**Semester:** Summer 2025<br>
**Instructor:** Majed Al Ghandour<br>
**Student:** Richard Hemphill<br>
**DCE ID:** 000970505<br>

## Instructions
For the lab, you will work with Google's API for Speech-to-text with an audio file you will create and Python Colab to transcribe your audio file into words with timestamps.

Follow the steps below to set up your Colab notebook for your lab, which will allow you to have your own copy of the lab to work from.

*   Save a copy in your Drive by selecting “File” and then “Save a copy in Drive.”
*   Name your file as YourLastName_FirstName_FinalProject.

**Submission:** Download as the .ipynb format and submit through canvas.

## **Step 1:** Authenticate the API requests by creating a service, then create credentials to authenticate as the service account by downloading a JSON file.

In [1]:
# setup environment
import os
!export GOOGLE_APPLICATION_CREDENTIALS=GOOGLE_APPLICATION_CREDENTIALS=school-project-466502-68f5a009b263.json
os.environ["GOOGLE_APPLICATION_CREDENTIALS"]="./school-project-466502-68f5a009b263.json"

## **Step 2:** Record your audio file with a simple statement in English.

In [2]:
# install extensions
!sudo apt install -qq ffmpeg
!pip install torchaudio ipywebrtc notebook --quiet requests
!jupyter nbextension enable --py widgetsnbextension

ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.8 MB/s eta 0:00:00
Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [3]:
# import libraries
from ipywebrtc import AudioRecorder, CameraStream
import torchaudio
from IPython.display import Audio

In [4]:
# enable widget in Colab
from google.colab import output
output.enable_custom_widget_manager()

In [6]:
# start widget for recording
from google.colab import output
output.enable_custom_widget_manager()
camera = CameraStream(constraints={'audio': True,'video':False})
recorder = AudioRecorder(stream=camera)
recorder

AudioRecorder(audio=Audio(value=b'', format='webm'), stream=CameraStream(constraints={'audio': True, 'video': …

In [7]:
# convert recording to wav format
with open('recording.webm', 'wb') as f:
    f.write(recorder.audio.value)
!ffmpeg -i recording.webm -ac 1 -f wav file.wav -y -hide_banner -loglevel panic
audio_file_path = 'file.wav'
sig, sr = torchaudio.load(audio_file_path)

# start widget for playback the wave file
Audio(data=sig, rate=sr)

In [8]:
# disable widgets in Colab
from google.colab import output
output.disable_custom_widget_manager()

## **Step 3:** Install the client library in Python Colab.

In [11]:
!pip3 install --user --upgrade google-cloud-speech

## **Step 4:** Import speech library into Python Colab

In [10]:
# import library
from google.cloud import speech_v1 as speech

ImportError: cannot import name 'speech_v1' from 'google.cloud' (unknown location)

## **Step 5:** Load your local audio file into the Python Colab or load it into your google bucket.

In [ ]:
# open the WAV file and record the audio
with open(audio_file_path, 'rb') as audio_file:
  content = audio_file.read()

## **Step 6:** Write a code to transcribe your audio file in Python Colab. Your output may be similar to this but with different confidence percentages.

In [ ]:
# instantiate a SpeechClient
client = speech.SpeechClient()

# pass audio content to Google Cloud
audio = speech.RecognitionAudio(content=content)
config = speech.RecognitionConfig(
    encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
    language_code="en-US",
    enable_word_time_offsets=True,  # Set to True for word-level confidence
)
response = client.recognize(config=config, audio=audio)

# show result
print(f"Transcript: {response.results[0].alternatives[0].transcript}")
print(f"Confidence: {response.results[0].alternatives[0].confidence:.0%}")

Transcript: hello my name is Richard hempill I enjoy cloud computing for machine learning
Confidence: 93%


## **Step 7:** Write a code to stamp your words from your audio file in Python Colab. Detect the time offsets (timestamps) for each word in your transcribed audio.

In [ ]:
# show start/stop time for each word spoken
for result in response.results:
    for word_info in result.alternatives[0].words:
        start_seconds = word_info.start_time.total_seconds()
        end_seconds = word_info.end_time.total_seconds()
        print(f"{start_seconds:0.3} | {end_seconds:0.3} | {word_info.word}")

0.8 | 2.9 | hello
2.9 | 3.6 | my
3.6 | 3.7 | name
3.7 | 3.9 | is
3.9 | 4.4 | Richard
4.4 | 5.0 | hempill
5.0 | 5.3 | I
5.3 | 5.6 | enjoy
5.6 | 6.4 | cloud
6.4 | 6.8 | computing
6.8 | 7.0 | for
7.0 | 7.8 | machine
7.8 | 8.0 | learning
